In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output
import random
import os
import environments_fully_observable 
import environments_partially_observable

from agent import Agent
import algorithms.actor_critic as ac
import algorithms.random as rn
import algorithms.zigzag as zz
import algorithms.greedy as gr

tf.random.set_seed(0)
random.seed(0)
np.random.seed(0)

In [ ]:
N_BOARDS = 1
GAMMA = 0.9

MODELS_PATH = "models"
MODEL_NAME = "actor_critic_separated_loss"

In [ ]:
def get_env(n=2):
    # n is the number of boards that you want to simulate parallely
    # size is the size of each board, also considering the borders
    # mask for the partially observable, is the size of the local neighborhood
    size = 11
    e = environments_fully_observable.OriginalSnakeEnvironment(n, size)
    # or environments_partially_observable.OriginalSnakeEnvironment(n, size, 2)
    return e
env_ = get_env(n=N_BOARDS)

In [ ]:
#ACTOR CRITIC
#logic = ac.create_logic(state_shape=env_.to_state().shape[1:], action_dim=4, optimizer=None, gamma=GAMMA)
#logic.load_models(folder_path=os.path.join(MODELS_PATH, MODEL_NAME, "best_model"), state_shape=env_.to_state().shape[1:], load_critic=False)
#agent = Agent(algorithm_logic=logic, algorithm_name=MODEL_NAME, algorithm_path=MODELS_PATH, save_frequency=500)

In [ ]:
#RANDOM
#logic = rn.create_logic(state_shape=env_.to_state().shape[1:], action_dim=4, n_boards=env_.n_boards, optimizer=None, gamma=GAMMA)
#agent = Agent(algorithm_logic=logic, algorithm_name="random", algorithm_path=MODELS_PATH, save_frequency=500)

In [ ]:
#ZIGZAG
logic = zz.create_logic(state_shape=env_.to_state().shape[1:], action_dim=4, n_boards=env_.n_boards, optimizer=None, gamma=GAMMA)
agent = Agent(algorithm_logic=logic, algorithm_name="zigzag", algorithm_path=MODELS_PATH, save_frequency=500)

In [ ]:
#GREEDY
#logic = gr.create_logic(state_shape=env_.to_state().shape[1:], action_dim=4, n_boards=env_.n_boards, optimizer=None, gamma=GAMMA)
#agent = Agent(algorithm_logic=logic, algorithm_name="greedy", algorithm_path=MODELS_PATH, save_frequency=500)

In [ ]:
state = tf.cast(env_.to_state(), tf.float32)
total_reward = 0
steps = 0
max_steps = 4000

for i in range(max_steps):
    # Use training=False to take the ARGMAX (deterministic best move)
    actions, _ = agent.get_action(state, training=False)
    
    # Environment expects 2D actions (n_boards, 1)
    if len(actions.shape) == 1:
        actions = tf.expand_dims(actions, axis=-1)
    
    #print(f"Actions taken: {actions.numpy().flatten()}")

    rewards = env_.move(actions)
    state = tf.cast(env_.to_state(), tf.float32)
    
    total_reward += np.sum(rewards)
    steps += 1
    
    # --- Optional: Visualization (works in Jupyter) ---
    clear_output(wait=True)
    # Get the raw board for visualization (Board 0)
    # 0=Wall, 1=Empty, 2=Fruit, 3=Body, 4=Head
    board_img = env_.boards[0] 
    
    plt.imshow(board_img, cmap='viridis', origin='lower')
    plt.title(f"Step: {steps} | Total Reward: {total_reward:.2f}")
    plt.show()
    
    

    # Check if the snake died (optional, depends on your env logic)
    if rewards[-1] < 0: # Hit wall or himself
        print("Game Over!")
        break
        
    time.sleep(0.01) # Slow down so you can see the movement

print(f"Evaluation finished in {steps} steps. Total Reward: {total_reward}")